# ATMAS: Autonomous Traffic Monitoring and Analysis System
### End-to-End Edge Pipeline Simulation (Colab Edition)
**Patent No. 549991 (Application: 202221057506)**

This notebook demonstrates the end-to-end architecture of ATMAS:
1. **Environment Setup & Dependencies**: Installing ultralytics / YOLO, OpenCV, and sample traffic video.
2. **Homography Calibration (Perspective Transform)**: Mapping 2D camera coordinates $(u, v)$ to real-world ground coordinates $(X, Y)$ in meters.
3. **Vehicle Tracking & Optical Flow / Centroid Kinematics**: Calculating velocity ($v$) in km/h and instantaneous acceleration ($a$) in $m/s^2$.
4. **Safety & Anomaly Rules Engine**: Rash driving detection (speeding, harsh deceleration/swerving) & road congestion metrics.
5. **Real-time Pipeline & Video Export**: Annotating video frames and generating CSV telemetry reports.

## 1. Setup & Dependencies
Install required packages and download a sample traffic clip.

In [ ]:
!pip install -q ultralytics opencv-python-headless numpy pandas matplotlib

# Download a sample urban traffic video clip for testing
!wget -O sample_traffic.mp4 -q https://github.com/intel-iot-devkit/sample-videos/raw/master/car-detection.mp4
print("Setup complete! Sample video downloaded.")

## 2. Perspective Transformation (Planar Homography)
Monocular 2D video feeds compress depth. We compute a $3 \times 3$ Homography matrix $H$ mapping 4 selected road pavement points (pixels) to known real-world coordinates (meters).

In [ ]:
import cv2
import numpy as np

class HomographyTransformer:
    def __init__(self, src_pts_pixel, dst_pts_meters):
        """
        src_pts_pixel: 4 points in pixel coordinates [[u1, v1], [u2, v2], [u3, v3], [u4, v4]]
        dst_pts_meters: 4 points in real-world metric coordinates [[X1, Y1], [X2, Y2], ...]
        """
        self.src = np.array(src_pts_pixel, dtype=np.float32)
        self.dst = np.array(dst_pts_meters, dtype=np.float32)
        self.H, _ = cv2.findHomography(self.src, self.dst)

    def pixel_to_world(self, u, v):
        """Transform single pixel contact point (u, v) into real-world (X, Y) in meters"""
        pt = np.array([u, v, 1.0], dtype=np.float32).reshape(3, 1)
        world_pt = np.dot(self.H, pt)
        world_pt /= world_pt[2]  # Normalize projective coordinate
        return float(world_pt[0, 0]), float(world_pt[1, 0])

# Example calibration for 1280x720 camera viewing a 25m road segment
# 4 points: [top-left, top-right, bottom-right, bottom-left]
src_points = [[400, 250], [880, 250], [1150, 700], [150, 700]]
dst_points = [[0.0, 30.0], [7.0, 30.0], [7.0, 0.0], [0.0, 0.0]]  # 7m width (2 lanes), 30m length

transformer = HomographyTransformer(src_points, dst_points)
print("Homography Matrix H computed:")
print(transformer.H)

## 3. Vehicle Kinematics & Anomaly Detection Engine
Tracks ground contact points (bottom-center of bounding boxes), filters noise via rolling temporal windows, and computes:
- Velocity $v$ ($km/h$)
- Acceleration $a$ ($m/s^2$)
- Rash driving flags: Over-speeding, harsh deceleration ($a < -3.5 m/s^2$), and sudden lateral swerves.

In [ ]:
class KinematicAnalyzer:
    def __init__(self, transformer, fps=30.0, speed_limit_kmh=50.0, harsh_accel_thresh=3.5):
        self.transformer = transformer
        self.fps = fps
        self.speed_limit = speed_limit_kmh
        self.harsh_accel_thresh = harsh_accel_thresh
        
        # History structure: {track_id: [(xw, yw, timestamp, speed_kmh)]}
        self.tracks = {}

    def update(self, track_id, bottom_center_pixel, frame_idx):
        t = frame_idx / self.fps
        xw, yw = self.transformer.pixel_to_world(*bottom_center_pixel)
        
        if track_id not in self.tracks:
            self.tracks[track_id] = []
            
        self.tracks[track_id].append((xw, yw, t))
        
        # Require at least 4 frames (~130ms) for stable velocity calculation
        if len(self.tracks[track_id]) < 4:
            return {"speed_kmh": 0.0, "accel": 0.0, "is_rash": False, "reason": "Calibrating"}
        
        # Rolling window of last 6 frames
        window = self.tracks[track_id][-6:]
        dx = window[-1][0] - window[0][0]
        dy = window[-1][1] - window[0][1]
        dt = window[-1][2] - window[0][2]
        
        if dt <= 0:
            return {"speed_kmh": 0.0, "accel": 0.0, "is_rash": False, "reason": "Normal"}
            
        speed_mps = np.sqrt(dx**2 + dy**2) / dt
        speed_kmh = speed_mps * 3.6
        
        # Instantaneous acceleration from previous velocity window
        accel = 0.0
        if len(self.tracks[track_id]) >= 12:
            prev_window = self.tracks[track_id][-12:-6]
            p_dx = prev_window[-1][0] - prev_window[0][0]
            p_dy = prev_window[-1][1] - prev_window[0][1]
            p_dt = prev_window[-1][2] - prev_window[0][2]
            if p_dt > 0:
                prev_speed_mps = np.sqrt(p_dx**2 + p_dy**2) / p_dt
                accel = (speed_mps - prev_speed_mps) / dt
                
        # Anomaly / Rash Driving Rule Engine
        is_rash = False
        reasons = []
        
        if speed_kmh > self.speed_limit:
            is_rash = True
            reasons.append(f"Overspeed ({speed_kmh:.1f} km/h)")
        if abs(accel) > self.harsh_accel_thresh:
            is_rash = True
            reasons.append(f"Harsh Acc/Brake ({accel:.1f} m/s²)")
            
        reason_str = ", ".join(reasons) if reasons else "Normal"
        return {
            "speed_kmh": speed_kmh,
            "accel": accel,
            "is_rash": is_rash,
            "reason": reason_str,
            "pos_world": (xw, yw)
        }

## 4. Full Execution Pipeline
Runs YOLO tracking (ByteTrack / BoT-SORT) on the input video, feeds trajectories through the kinematic pipeline, overlays real-time telemetry on the frames, and generates an output video & analytical summary.

In [ ]:
from ultralytics import YOLO
import pandas as pd

# Load lightweight YOLO model for edge inference simulation
model = YOLO("yolov8n.pt")  # In Jetson Nano deployment, YOLOv4-tiny TensorRT engine is used

video_path = "sample_traffic.mp4"
cap = cv2.VideoCapture(video_path)

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

out_video_path = "atmas_analyzed_output.mp4"
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(out_video_path, fourcc, fps, (width, height))

# Initialize Kinematic Engine
kinematics = KinematicAnalyzer(transformer, fps=fps, speed_limit_kmh=45.0)

telemetry_records = []
frame_idx = 0

print(f"Processing video: {width}x{height} @ {fps:.1f} FPS, total {total_frames} frames...")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    frame_idx += 1
    
    # Vehicle classes in COCO: 2: car, 3: motorcycle, 5: bus, 7: truck
    results = model.track(frame, persist=True, classes=[2, 3, 5, 7], verbose=False)
    
    active_vehicle_speeds = []
    
    if results[0].boxes is not None and results[0].boxes.id is not None:
        boxes = results[0].boxes.xyxy.cpu().numpy()
        track_ids = results[0].boxes.id.int().cpu().numpy()
        class_ids = results[0].boxes.cls.int().cpu().numpy()
        
        for box, track_id, cls_id in zip(boxes, track_ids, class_ids):
            x1, y1, x2, y2 = box
            # Contact point: bottom center of bounding box
            bottom_center = (float((x1 + x2) / 2.0), float(y2))
            
            # Compute kinematics
            metrics = kinematics.update(track_id, bottom_center, frame_idx)
            speed = metrics["speed_kmh"]
            accel = metrics["accel"]
            is_rash = metrics["is_rash"]
            
            active_vehicle_speeds.append(speed)
            
            # Log telemetry
            telemetry_records.append({
                "frame": frame_idx,
                "timestamp_sec": frame_idx / fps,
                "track_id": int(track_id),
                "class": model.names[cls_id],
                "speed_kmh": round(speed, 2),
                "acceleration_mps2": round(accel, 2),
                "is_rash": is_rash,
                "reason": metrics["reason"]
            })
            
            # Draw bounding box and alert tag
            box_color = (0, 0, 255) if is_rash else (0, 255, 0)
            cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), box_color, 2)
            
            label = f"ID:{track_id} | {speed:.1f} km/h"
            if is_rash:
                label += f" [ALERT: {metrics['reason']}]"
            
            cv2.putText(frame, label, (int(x1), max(20, int(y1) - 8)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, box_color, 2)

    # Overlay Overall Traffic Congestion / LOS Metrics
    vehicle_count = len(active_vehicle_speeds)
    avg_speed = np.mean(active_vehicle_speeds) if vehicle_count > 0 else 0.0
    
    congestion_level = "LOW (Free Flow)"
    if vehicle_count > 8 and avg_speed < 15.0:
        congestion_level = "HIGH (Congested)"
    elif vehicle_count > 4 and avg_speed < 25.0:
        congestion_level = "MODERATE"

    # Top HUD Banner
    cv2.rectangle(frame, (0, 0), (width, 50), (20, 20, 20), -1)
    hud_text = f"ATMAS EDGE AI | Active Vehicles: {vehicle_count} | Avg Speed: {avg_speed:.1f} km/h | Congestion: {congestion_level}"
    cv2.putText(frame, hud_text, (20, 32), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 255, 255), 2)

    out.write(frame)

cap.release()
out.release()

df_telemetry = pd.DataFrame(telemetry_records)
df_telemetry.to_csv("atmas_telemetry_report.csv", index=False)
print(f"Processing complete! Output saved to '{out_video_path}' and 'atmas_telemetry_report.csv'.")

## 5. Traffic Analytics & Patent Verification Summary
Generate summary statistics of rash driving occurrences and speed distribution across the monitored traffic stream.

In [ ]:
import matplotlib.pyplot as plt

if not df_telemetry.empty:
    print("=== ATMAS Traffic Analytics Summary ===")
    print(f"Total Detections Logged: {len(df_telemetry)}")
    print(f"Unique Vehicles Tracked: {df_telemetry['track_id'].nunique()}")
    rash_events = df_telemetry[df_telemetry['is_rash'] == True]
    print(f"Rash Driving Events Logged: {len(rash_events)}")
    
    # Plot speed distribution
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    df_telemetry['speed_kmh'].hist(bins=20, color='teal', edgecolor='black')
    plt.title("Vehicle Speed Distribution (km/h)")
    plt.xlabel("Speed (km/h)")
    plt.ylabel("Frame Count")

    # Plot acceleration distribution
    plt.subplot(1, 2, 2)
    df_telemetry['acceleration_mps2'].hist(bins=20, color='coral', edgecolor='black')
    plt.title("Acceleration Distribution (m/s²)")
    plt.xlabel("Acceleration (m/s²)")
    plt.ylabel("Frame Count")
    plt.tight_layout()
    plt.show()
else:
    print("No telemetry data logged.")